# 02 · Python II: Functions & Data Structures

This is the hands-on companion to [`index.html`](index.html). It assumes you have completed
**Module 01** and nothing more.

By the end, you will be able to:

- write and call functions with parameters, arguments, defaults, and return values;
- explain the difference between returning a value and printing it;
- predict which name is visible where, using local scope and shadowing;
- build, index, slice, and update lists, tuples, dictionaries, and sets;
- choose the right container for a job and justify the choice;
- explain aliasing, `is` vs `==`, and shallow vs deep copies;
- rewrite loops as comprehensions and back again; and
- assemble small functions into one working multi-learner report.

**Working rule:** predict first, run second, explain third. A wrong prediction is useful—it
shows exactly which mental model needs adjusting.

## 0 · How to use this notebook

Same as Module 01: click a code cell, press **Shift + Enter** to run it and move on. Output
appears below the cell. If the notebook behaves strangely because cells were run out of
order, use **Kernel → Restart Kernel and Run All Cells**.

Run this notebook from top to bottom the first time. No third-party packages are required.

> Cells marked **Predict before running** contain a comment line for your guess. Fill it in
> before you run the cell. That habit is the single fastest way to learn.

In [ ]:
# A quick Module 01 refresher: values, an f-string, a loop, and a condition.
learner = "Maya"
minutes = [30, 45, 20]

total = 0
for value in minutes:
    total = total + value

if total >= 60:
    verdict = "a solid start"
else:
    verdict = "keep going"

print(f"{learner} studied {total} minutes: {verdict}.")
print("Notebook connected: Python is ready.")

That cell used everything Module 01 taught. Notice the awkward parts: the totalling loop has
to be rewritten every time you need it, and `minutes` had to be a list—which Module 01 never
explained. Both gaps close in this module.

## 1 · Defining, calling, and returning

A **function** is a named block of code that does one job. `def` creates it; parentheses run
it. Two separate moments: **definition** (nothing runs) and **call** (the body runs).

In [ ]:
def to_fahrenheit(celsius):
    """Convert a Celsius temperature to Fahrenheit."""
    return celsius * 9 / 5 + 32

print("The def statement above produced no output -- it only created the function.")
print(to_fahrenheit(20))
print(to_fahrenheit(100))
print(to_fahrenheit(-40))

One definition, three calls. Change the formula in one place and every call is corrected.

The line in triple quotes is a **docstring**: the function's own documentation. `help()`
displays it, which is exactly what you read when you look up a library function.

In [ ]:
help(to_fahrenheit)
print(to_fahrenheit.__doc__)

### Predict before running · returning vs printing

This is the most common beginner confusion about functions. Write your prediction for each
`print` below, then run the cell.

In [ ]:
# My prediction: value_a = ___  and  value_b = ___

def double_print(n):
    print(n * 2)          # displays, gives nothing back

def double_return(n):
    return n * 2          # hands a value back

value_a = double_print(5)
value_b = double_return(5)

print("value_a is", value_a, type(value_a))
print("value_b is", value_b, type(value_b))
print("value_b + 1 is", value_b + 1)
# print(value_a + 1)   # uncomment to see: TypeError, because None is not a number

A function that never reaches a `return` gives back `None`. `None` cannot be used in
arithmetic—which is why the commented line would fail.

`return` also **exits the function immediately**. That makes early returns a clean way to
handle special cases.

In [ ]:
def safe_divide(a, b):
    """Return a / b, or 0.0 when b is zero."""
    if b == 0:
        return 0.0        # leaves the function right here
    return a / b

print(safe_divide(10, 4))
print(safe_divide(10, 0))

## 2 · Parameters, arguments, and defaults

- A **parameter** is a name in the definition: the blank to be filled.
- An **argument** is a value in the call: what fills the blank.

Arguments match parameters by position unless you name them.

In [ ]:
def area(width, height):
    """Return the area of a rectangle."""
    return width * height

print(area(3, 2))                 # positional
print(area(width=3, height=2))    # keyword
print(area(height=2, width=3))    # keyword, any order
print(area(3, height=2))          # positional first, then keyword

# print(area(width=3, 2))   # uncomment: SyntaxError -- positional after keyword

A **default value** makes a parameter optional. Parameters with defaults must come last.

In [ ]:
def greet(name, greeting="Hello", punctuation="!"):
    """Return a greeting line for name."""
    return f"{greeting}, {name}{punctuation}"

print(greet("Maya"))
print(greet("Maya", "Welcome"))
print(greet("Maya", punctuation="?"))     # skips greeting -- only a keyword can do this

### The mutable default argument trap

A default value is created **once, at definition time**. With a list as the default, every
call that omits the argument shares the *same* list. Run this and watch the bug happen.

In [ ]:
def collect_bad(value, items=[]):      # ONE list, created when def ran
    items.append(value)
    return items

print("bad :", collect_bad(1))
print("bad :", collect_bad(2))         # surprise: the 1 is still there
print("bad :", collect_bad(3))

def collect_good(value, items=None):
    if items is None:
        items = []                     # a fresh list on every call
    items.append(value)
    return items

print("good:", collect_good(1))
print("good:", collect_good(2))

# Proof of the cause: the default list is stored on the function object itself.
print("stored default:", collect_bad.__defaults__)

**Rule:** default values must be immutable. Use `None` as the signal for "make a fresh one".

### Accepting any number of arguments

`*args` collects extra positional arguments into a tuple. You mainly need to *recognise*
this when reading documentation.

In [ ]:
def total_of(*numbers):
    """Return the sum of every argument given."""
    running = 0
    for n in numbers:
        running += n
    return running

print(total_of(1, 2))
print(total_of(1, 2, 3, 4, 5))
print(total_of())

### Returning several values at once

Commas in a `return` pack the values into one **tuple**, which the caller can **unpack**.

In [ ]:
def split_time(total_minutes):
    """Return whole hours and remaining minutes."""
    return total_minutes // 60, total_minutes % 60

hours, minutes = split_time(137)        # unpacked into two names
print(f"{hours} h {minutes} min")

both = split_time(137)                  # or kept as one tuple
print(both, type(both))

# hours, minutes, seconds = split_time(137)   # uncomment: ValueError -- counts must match

## 3 · Scope: where a name lives

Names created inside a function are **local**: they exist only during that call. Names at
the top level are **global**. A function may *read* a global, but assigning to a name makes
it local for the whole call.

In [ ]:
tax_rate = 0.05          # global

def with_tax(price):
    """Return price plus tax, reading the global rate."""
    subtotal = price     # subtotal is local to this call
    return subtotal * (1 + tax_rate)

print(with_tax(100))
# print(subtotal)        # uncomment: NameError -- subtotal did not survive the call

### Predict before running · shadowing

The same spelling, `total`, exists globally and locally. Which value does each `print` show?

In [ ]:
# My prediction: inside = ___   outside = ___   result = ___

total = 100

def add_ten(total):          # this parameter shadows the global name
    total = total + 10
    print("inside :", total)
    return total

result = add_ten(5)
print("outside:", total)     # the global was never touched
print("result :", result)

The fix for "I need to update a global" is almost always: pass the value in, return the new
one, and reassign at the top level.

In [ ]:
def bump(count):
    """Return count increased by one."""
    return count + 1

count = 0
count = bump(count)
count = bump(count)
print(count)

# Shadowing a BUILT-IN name is the version that really hurts:
#     sum = 10
#     sum([1, 2])     # TypeError: 'int' object is not callable
# Leave those two lines commented. If you ever do it by accident,
# `del sum` restores the built-in, or restart the kernel.
print(sum([1, 2]))   # still the real built-in

## 4 · Lists, slicing, and sequence tools

A **list** is an ordered, changeable collection. Indices count from 0; negative indices
count from the end.

In [ ]:
scores = [90, 85, 77, 92]

print(scores, "has", len(scores), "elements")
print("first :", scores[0])
print("last  :", scores[-1])
print("second from the end:", scores[-2])
# print(scores[4])    # uncomment: IndexError -- valid indices are 0..3

Lists are **mutable**: they can be changed in place. Watch the same list evolve.

In [ ]:
scores = [90, 85, 77, 92]

scores[0] = 95                 # replace one element
print("after replace:", scores)

scores.append(60)              # add one at the end
print("after append :", scores)

scores.extend([70, 65])        # add every element of another collection
print("after extend :", scores)

removed = scores.pop()         # remove AND return the last element
print("popped", removed, "->", scores)

scores.remove(77)              # delete the first element equal to 77
print("after remove :", scores)

print("count of 95:", scores.count(95), " index of 85:", scores.index(85))

### The `.sort()` versus `sorted()` trap

Methods that change a list in place return `None`. This is the most common list bug in
beginner code.

In [ ]:
values = [3, 1, 2]
broken = values.sort()          # sorts in place, returns None
print("values:", values, " broken:", broken)

values = [3, 1, 2]
ordered = sorted(values)        # new list; the original is untouched
print("values:", values, " ordered:", ordered)
print("descending:", sorted(values, reverse=True))

### Predict before running · slicing

`sequence[start:stop:step]` includes the start and **excludes the stop**—the same rule as
`range()`. Write each guess as a comment first.

In [ ]:
word = "TRANSFORMER"

print(word[0:5])     # guess:
print(word[:3])      # guess:
print(word[-3:])     # guess:
print(word[::2])     # guess:
print(word[::-1])    # guess:  (a negative step reads backwards)
print(word[3:1])     # guess:  (stop before start)
print(word[0:99])    # guess:  (out of range -- error, or something else?)

Slicing works identically on strings, lists, and tuples, because all three are
**sequences**. A slice never raises `IndexError`; it clamps to what exists.

These built-ins work on any sequence:

In [ ]:
temps = [18, 22, 25, 19, 30, 27, 21]

print("len  :", len(temps))
print("sum  :", sum(temps))
print("min  :", min(temps), " max:", max(temps))
print("mean :", sum(temps) / len(temps))
print("30 in temps?", 30 in temps)
print("sorted:", sorted(temps))

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

for position, day in enumerate(days):        # position AND value
    print(position, day, end="   ")
print()

for day, temp in zip(days, temps):           # two collections in step
    print(f"{day}: {temp}", end="   ")
print()

### The two functions that connect strings and lists

`.split()` turns a string into a list; `.join()` turns a list of strings into one string.
Nearly all text processing starts here, including tokenization in the LLM modules.

In [ ]:
sentence = "machine learning is applied statistics"

words = sentence.split()
print(words)
print(len(words), "words")
print(" | ".join(words))
print("a,b,,c".split(","))          # a chosen separator; note the empty piece

# The separator owns .join(), and every element must already be a string:
numbers = [1, 2, 3]
print("-".join(str(n) for n in numbers))
# print("-".join(numbers))    # uncomment: TypeError -- ints are not strings

### Nested lists

An element can itself be a list. This is the plain-Python ancestor of NumPy's 2-D arrays
(Module 07) and a DataFrame's rows (Module 08).

In [ ]:
grid = [[1, 2, 3],
        [4, 5, 6]]

print("row 0      :", grid[0])
print("row 1 col 2:", grid[1][2])
print("rows:", len(grid), " columns:", len(grid[0]))

for row in grid:
    for value in row:
        print(value, end=" ")
    print()

## 5 · Tuples, dictionaries, and sets

A **tuple** is an ordered collection that cannot be changed after creation.

In [ ]:
point = (3, 1)
rgb = (255, 128, 0)
one_item = (5,)          # the COMMA makes it a tuple
not_a_tuple = (5)        # just the integer 5 in parentheses

print(point, point[0], point[-1], len(rgb))
print(type(one_item), type(not_a_tuple))
# point[0] = 9           # uncomment: TypeError -- tuples do not support item assignment

# Everything that only READS a sequence still works:
print(sorted(rgb), sum(rgb), 128 in rgb, rgb[:2])

Tuples announce "this shape is fixed", are safe from accidental change, and—being
**hashable**—can be dictionary keys or set elements. Lists cannot.

**Unpacking** assigns elements to several names at once.

In [ ]:
point = (3, 1)
x, y = point
print(x, y)

a, b = 1, 2
a, b = b, a              # swap, no temporary variable needed
print(a, b)

first, *rest = [10, 20, 30, 40]      # * collects the remainder into a list
print(first, rest)

for index, letter in enumerate("ML"):    # enumerate hands you tuples
    print(index, letter)

### Dictionaries

A **dictionary** maps unique **keys** to **values**. A list answers "what is at position 3?";
a dict answers "what belongs to `'ada'`?".

In [ ]:
ages = {"ada": 36, "alan": 41, "grace": 45}

print(ages["ada"])            # look up by key
ages["ada"] = 37              # update an existing key
ages["katherine"] = 52        # writing a NEW key silently creates it
del ages["alan"]              # remove a key

print(ages)
print(len(ages), "entries")
print("'grace' in ages:", "grace" in ages)      # membership tests KEYS
print("45 in ages:", 45 in ages)                # not values -- False

Reading a missing key is an error; `.get()` gives you a fallback instead.

In [ ]:
# print(ages["bob"])          # uncomment: KeyError: 'bob'
print(ages.get("bob"))        # None -- no error
print(ages.get("bob", 0))     # 0 -- your chosen fallback
print(ages)                   # .get() did NOT add anything

In [ ]:
# Three ways to loop over a dict.
for name in ages:                       # keys, by default
    print(name, end=" ")
print()

print("values:", list(ages.values()), " sum:", sum(ages.values()))

for name, age in ages.items():          # both halves at once -- prefer this
    print(f"{name} is {age}")

print("keys, sorted:", sorted(ages))

### The word counter

Counting occurrences is the classic dictionary program—and the ancestor of vocabulary
building in the language-model modules. Here are both versions from the lesson.

In [ ]:
words = ["the", "cat", "sat", "on", "the", "mat", "the", "end"]

# Explicit version: good for learning.
counts = {}
for word in words:
    if word in counts:
        counts[word] = counts[word] + 1
    else:
        counts[word] = 1
print(counts)

# Idiomatic version: identical result, one line.
counts_2 = {}
for word in words:
    counts_2[word] = counts_2.get(word, 0) + 1
print(counts_2)
print("same result?", counts == counts_2)

### Nested data

Two shapes cover most real data, and both appear in this module's project.

In [ ]:
study_log = {                       # dict of lists
    "maya": [30, 45, 20],
    "sam": [60, 60],
}
print(study_log["maya"][1])         # read left to right: the list, then index 1
print(sum(study_log["sam"]))

learners = {                        # dict of dicts
    "maya": {"minutes": 95, "days": 3},
    "sam": {"minutes": 120, "days": 2},
}
print(learners["sam"]["minutes"])

for name, minutes in study_log.items():
    print(f"{name}: {len(minutes)} days, {sum(minutes)} minutes")

### Sets

A **set** holds unique, hashable values with no order. Its two jobs are removing duplicates
and answering "is this present?" quickly.

In [ ]:
tags = {"ml", "python", "ml"}        # the duplicate simply does not appear
print(tags, len(tags))

tags.add("stats")
tags.discard("python")               # .remove() would raise KeyError if absent
print(tags, "'ml' in tags:", "ml" in tags)

words = ["the", "cat", "the", "mat"]
print("unique, sorted:", sorted(set(words)))
# print(tags[0])                     # uncomment: TypeError -- a set has no order to index

# {} is an empty DICT, not an empty set:
print(type({}), type(set()))

In [ ]:
done = {"01", "02", "03"}
required = {"02", "03", "04"}

print("both      :", done & required)      # intersection
print("still to do:", required - done)     # difference
print("either    :", done | required)      # union
print("exactly one:", done ^ required)     # symmetric difference
print("subset?   :", {"02"} <= required)

## 6 · Mutability and aliasing, proved with `id()`

Assignment **never copies**. `b = a` attaches a second name to the same object. `id(x)`
returns an object's identity, so you can prove it.

In [ ]:
a = [1, 2, 3]
b = a            # alias: the same object
c = a.copy()     # copy: a new object

print("a is b:", a is b, "  a is c:", a is c)
print("a == c:", a == c, "  (equal contents, different objects)")
print("id(a):", id(a))
print("id(b):", id(b), "<- identical to a")
print("id(c):", id(c), "<- different")

b.append(99)
print("after b.append(99):  a =", a, "  c =", c)

`==` asks "equal contents?"; `is` asks "the same one object?". Use `==` for values, and
reserve `is` for `None`.

Now the difference between rebinding a name and mutating an object:

In [ ]:
# Immutable: the names come apart, because a new object is built.
x = 5
y = x
y = y + 1
print("ints :", x, y)

# Mutable: the names stay together, because the object itself changed.
p = [1, 2, 3]
q = p
q.append(4)
print("lists:", p, q)

### Shallow versus deep copies

`a.copy()`, `a[:]`, and `list(a)` all make a **shallow** copy: a new outer list holding the
*same* inner objects.

In [ ]:
import copy

grid = [[1, 2], [3, 4]]
shallow = grid.copy()
deep = copy.deepcopy(grid)

shallow[0].append(99)      # mutates an inner list that grid also sees
print("after shallow change:", grid)

deep[1].append(77)         # fully independent
print("after deep change   :", grid)

print("inner shared?", grid[0] is shallow[0], " deep shared?", grid[0] is deep[0])

### Never mutate a collection while looping over it

In [ ]:
numbers = [1, 2, 3, 4]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)       # positions shift underneath the loop
print("buggy result:", numbers, "  (2 was removed, 4 was skipped)")

# Two safe fixes:
numbers = [1, 2, 3, 4]
for n in numbers[:]:            # iterate over a copy
    if n % 2 == 0:
        numbers.remove(n)
print("copy fix   :", numbers)

numbers = [1, 2, 3, 4]
numbers = [n for n in numbers if n % 2 != 0]     # better: build a new list
print("rebuild fix:", numbers)

### Passing a mutable object to a function

The parameter becomes another name for the caller's object—so a function can change the
caller's list. Be deliberate, and say which you do in the docstring.

In [ ]:
def add_bonus_in_place(scores):
    """Append a bonus to the caller's list. Mutates its argument."""
    scores.append(100)

def with_bonus(scores):
    """Return a NEW list with a bonus added. Leaves the argument alone."""
    return scores + [100]

original = [90, 85]
add_bonus_in_place(original)
print("after in-place :", original)

original = [90, 85]
new = with_bonus(original)
print("after pure call:", original, " new:", new)

## 7 · Comprehensions and the built-in toolkit

A **comprehension** builds a collection from another in one expression. Change the brackets
to change what you build.

In [ ]:
names = ["ada", "turing", "hopper", "bo"]

squares      = [n ** 2 for n in range(1, 6)]
lengths      = [len(w) for w in names]
lengths_set  = {len(w) for w in names}          # duplicates dropped
by_name      = {w: len(w) for w in names}       # key: value
long_names   = [w for w in names if len(w) > 2]  # filtered
upper_names  = [w.upper() for w in names]

print(squares)
print(lengths)
print(lengths_set)
print(by_name)
print(long_names)
print(upper_names)

Read a comprehension in execution order: "for each `w` in names, if it passes the filter,
put this expression in the new collection."

### Generator expressions

Round brackets compute values one at a time instead of building a whole collection. When a
function consumes them immediately, drop the brackets entirely.

In [ ]:
scores = [58, 91, 74, 45]

print(sum(n for n in scores if n >= 60))     # no intermediate list built
print(any(n < 50 for n in scores))           # is at least one true?
print(all(n < 50 for n in scores))           # are they all true?
print(max(len(w) for w in names))

### Sorting by something other than the value

`sorted()` takes a `key` argument: a **function** applied to each element to get the value
to sort by. The function is passed **without parentheses**—you are handing over the function
itself, not calling it.

In [ ]:
print(sorted(names))                      # alphabetical
print(sorted(names, key=len))             # shortest first

counts = {"the": 3, "cat": 1, "mat": 2}
print(sorted(counts, key=counts.get, reverse=True))    # keys, most frequent first
print(max(counts, key=counts.get))                     # the single most frequent key

# A lambda is a one-expression function with no name -- handy for a key used once.
learners = [("maya", 155), ("sam", 120), ("ren", 30)]
print(sorted(learners, key=lambda pair: pair[1], reverse=True))
print(sorted(learners, key=lambda pair: pair[0]))

# In Python a function is an ordinary value you can store and pass around:
chosen = len
print(chosen("hopper"))

### When *not* to use a comprehension

In [ ]:
# Poor style: the list is thrown away; the code exists for the printing side effect.
throwaway = [print(w) for w in ["a", "b"]]
print("the comprehension built:", throwaway)

# Honest version:
for w in ["a", "b"]:
    print(w)

## 8 · Guided practice with gentle checks

Replace each `...` or `None` with your answer. Every check reports **not attempted** until
you do, so the notebook stays safe to run from top to bottom.

Do not scroll to the solutions until you have wrestled with each exercise.

### First, the machinery — which you can now read

Module 01 asked you to run a `check` cell without studying it. You now know everything in
it. Read it before running:

- `def check(...)` defines a function with three **parameters**;
- `expected` and `tolerance` have **default values**, so callers may omit them;
- `if actual is None` uses `is` for a `None` comparison, exactly as Part 6 advised;
- the `isinstance` branch compares floats with a small tolerance, because `0.1 + 0.2` is not
  exactly `0.3` (Module 01's floating-point deep dive); and
- it **prints** rather than returning, because its whole job is a side effect: telling you
  how you did.

In [ ]:
def check(name, actual, expected, tolerance=1e-9):
    """Print whether `actual` matches `expected`. Treats None as not attempted."""
    if actual is None:
        print(f"o {name}: not attempted yet")
    elif isinstance(expected, float) and isinstance(actual, (int, float)):
        if abs(actual - expected) <= tolerance:
            print(f"+ {name}: correct")
        else:
            print(f"x {name}: got {actual}; expected {expected}")
    elif actual == expected:
        print(f"+ {name}: correct")
    else:
        print(f"x {name}: got {actual!r}; expected {expected!r}")

check("Self-test A", 4, 4)
check("Self-test B", None, 4)
check("Self-test C", 5, 4)

### Exercise 1 · Write a function

Define `rectangle_area(width, height)` with a docstring that **returns** the area.
Replace the `...` with your code—remember to return, not print.

In [ ]:
def rectangle_area(width, height):
    """Return the area of a rectangle."""
    ...        # replace this line

check("Exercise 1", rectangle_area(3, 4), 12)

### Exercise 2 · A default argument

Define `apply_discount(price, percent=10)` returning the price after the discount.
`apply_discount(200)` must give `180.0`, and `apply_discount(200, 25)` must give `150.0`.

In [ ]:
def apply_discount(price, percent=10):
    """Return price reduced by percent."""
    ...        # replace this line

check("Exercise 2a", apply_discount(200), 180.0)
check("Exercise 2b", apply_discount(200, 25), 150.0)

### Exercise 3 · Return two values

Define `min_max(values)` returning the smallest and largest, as two values separated by a
comma. Do not use the built-in `min()` or `max()`—use the best-so-far pattern, which is
exactly how model selection works later in the course.

In [ ]:
def min_max(values):
    """Return the smallest and largest value in a list."""
    ...        # replace this line

check("Exercise 3", min_max([7, 2, 9, 4]), (2, 9))

### Exercise 4 · Scope

Predict, without running anything, what the **global** `label` holds after this code:

```python
label = "global"

def relabel():
    label = "local"
    return label

relabel()
```

Set `exercise_4` to that value, as a string.

In [ ]:
exercise_4 = None      # replace None with "global" or "local"

check("Exercise 4", exercise_4, "global")

### Exercise 5 · Slicing

Using one slice expression, set `exercise_5` to every second temperature in `temps`,
starting from the first.

In [ ]:
temps = [18, 22, 25, 19, 30, 27, 21]

exercise_5 = None      # replace None with a slice of temps

check("Exercise 5", exercise_5, [18, 25, 30, 21])

### Exercise 6 · sorted() versus .sort()

Set `exercise_6` to the **hottest** temperature using `sorted()` and an index—not `max()`.
Afterwards, `temps` must still be in its original order, which the second check confirms.

In [ ]:
exercise_6 = None      # replace None

check("Exercise 6", exercise_6, 30)
check("Exercise 6 · temps unchanged", temps, [18, 22, 25, 19, 30, 27, 21])

### Exercise 7 · Count with a dict

Build a dict mapping each letter of `"mississippi"` to how many times it appears. Use
`.get()` with a fallback so the first occurrence works.

In [ ]:
exercise_7 = None      # replace None with {} and build it up in a loop

# your loop here

check("Exercise 7", exercise_7, {"m": 1, "i": 4, "s": 4, "p": 2})

### Exercise 8 · Deduplicate with a set

Set `exercise_8` to the unique words of `raw_words`, sorted alphabetically, as a **list**.

In [ ]:
raw_words = ["the", "cat", "the", "mat", "cat"]

exercise_8 = None      # replace None

check("Exercise 8", exercise_8, ["cat", "mat", "the"])

### Exercise 9 · Aliasing

Predict, without running it, what `x` holds after this code:

```python
x = [1, 2]
y = x
z = x[:]
y.append(3)
```

Set `exercise_9` to that value.

In [ ]:
exercise_9 = None      # replace None with the predicted list

check("Exercise 9", exercise_9, [1, 2, 3])

### Exercise 10 · A list comprehension

Rewrite this loop as one comprehension:

```python
result = []
for n in range(1, 11):
    if n % 2 == 0:
        result.append(n ** 2)
```

In [ ]:
exercise_10 = None     # replace None with one comprehension

check("Exercise 10", exercise_10, [4, 16, 36, 64, 100])

### Exercise 11 · A dict comprehension and a sort key

Two parts. Build a dict mapping each name in `people` to its length. Then set
`exercise_11b` to the names sorted from longest to shortest.

In [ ]:
people = ["ada", "turing", "hopper", "bo"]

exercise_11a = None    # replace None with a dict comprehension
exercise_11b = None    # replace None using sorted(..., key=..., reverse=...)

check("Exercise 11a", exercise_11a, {"ada": 3, "turing": 6, "hopper": 6, "bo": 2})
check("Exercise 11b", exercise_11b, ["turing", "hopper", "ada", "bo"])

## 9 · Mini-project: the study-log analyzer

Module 01's project reported on one learner with a single loop. This one reports on many
learners and is assembled from small functions—the growth this module is about.

Build it milestone by milestone. Every calculation function is **pure**: it reads its
arguments and returns a value. Only `print_report` has a side effect.

In [ ]:
# ---- Milestones 1-3: one learner at a time -------------------------------
def average_minutes(minutes):
    """Return the mean daily minutes, or 0.0 for an empty list."""
    if not minutes:
        return 0.0
    return sum(minutes) / len(minutes)


def classify(average):
    """Return 'strong', 'building', or 'start small' for an average."""
    if average >= 45:
        return "strong"
    if average >= 20:
        return "building"
    return "start small"


def days_at_goal(minutes, goal=30):
    """Return how many days met or exceeded the goal."""
    return len([m for m in minutes if m >= goal])


def summarize(minutes, goal=30):
    """Return one learner's figures as a dict."""
    average = average_minutes(minutes)
    return {
        "days": len(minutes),
        "total": sum(minutes),
        "average": average,
        "habit": classify(average),
        "at_goal": days_at_goal(minutes, goal),
    }


# Test the pieces BEFORE building on them -- this is the whole point of functions.
check("average of [30, 45, 20, 60]", average_minutes([30, 45, 20, 60]), 38.75)
check("average of []", average_minutes([]), 0.0)
check("classify(45) boundary", classify(45), "strong")
check("classify(20) boundary", classify(20), "building")
check("classify(19.9)", classify(19.9), "start small")
check("days_at_goal default", days_at_goal([30, 45, 20, 60]), 3)
check("days_at_goal goal=45", days_at_goal([30, 45, 20, 60], 45), 2)

In [ ]:
# ---- Milestones 4-5: the whole log --------------------------------------
def build_report(log, goal=30):
    """Return {name: summary} for every learner in the log."""
    return {name: summarize(minutes, goal) for name, minutes in log.items()}


def top_learner(report):
    """Return (name, total_minutes) for the learner with the most minutes."""
    if not report:
        return "nobody", 0
    best_name = max(report, key=lambda name: report[name]["total"])
    return best_name, report[best_name]["total"]


def print_report(report):
    """Display the report. The only function here with a side effect."""
    print("STUDY LOG REPORT")
    print("=" * 46)
    ranked = sorted(report, key=lambda name: report[name]["total"], reverse=True)
    for name in ranked:
        s = report[name]
        print(f"{name:<8}{s['days']:>3} days  {s['total']:>4} min  "
              f"avg {s['average']:>5.1f}  {s['habit']}")
    print("-" * 46)
    best_name, best_total = top_learner(report)
    print(f"Top learner: {best_name} with {best_total} minutes")
    total_all = sum(s["total"] for s in report.values())
    print(f"Class total: {total_all} minutes across {len(report)} learners")


study_log = {
    "maya": [30, 45, 20, 60],
    "sam": [60, 60],
    "ren": [10, 15, 5],
}

report = build_report(study_log)
print_report(report)
print()
check("top_learner", top_learner(report), ("maya", 155))
check("empty report", top_learner({}), ("nobody", 0))

### Project extensions

Work through these in the cell below. Each one is a small function plus one call—exactly the
habit the rest of the course depends on.

1. `best_day(minutes)` — the largest single day, or `0` for an empty list.
2. `class_average(report)` — the mean of every learner's average, rounded to one decimal.
3. `needs_encouragement(report)` — a **set** of names whose habit is `"start small"`.
4. Add a fourth learner to `study_log` and confirm the report includes them with no other
   edit.
5. Make `summarize()` ignore negative minutes by filtering them out first.

In [ ]:
# Your extensions here. Uncomment the checks as you complete each one.

def best_day(minutes):
    """Return the largest single day's minutes, or 0 for an empty list."""
    ...

def class_average(report):
    """Return the mean of every learner's average, rounded to one decimal."""
    ...

def needs_encouragement(report):
    """Return the set of names whose habit is 'start small'."""
    ...

check("best_day", best_day([30, 45, 20, 60]), 60)
check("best_day empty", best_day([]), 0)
check("class_average", class_average(report), 36.2)   # not 36.3 -- see the note in solutions
check("needs_encouragement", needs_encouragement(report), {"ren"})

### From notebook to script

Open [`study_stats.py`](study_stats.py) in this folder and run it from a terminal:

```
cd modules\02-python-functions-data-structures
python study_stats.py
```

It prints the report for a built-in sample log, then offers to collect your own learners.
Notice three things:

- the definitions come first, the data second, and one call starts everything;
- `if __name__ == "__main__":` at the bottom means "run this only when the file is executed
  directly, not when it is imported"—Module 03 covers importing properly; and
- each run starts with fresh state, unlike this notebook.

## 10 · Optional challenges

Attempt these only after the exercises and the project. Each combines several ideas.

### Challenge A · `top_word(words)`

Return the most frequent word in a list, reusing your counter from Exercise 7 and the
best-so-far pattern from Exercise 3. Then try `max(counts, key=counts.get)` and explain
why it works.

In [ ]:
# Write Challenge A here.

### Challenge B · Fix the grammar

`print_report` prints `1 days` for a learner with a single day. Write
`day_label(count)` returning `"1 day"` or `"3 days"` as appropriate, and use it. Then
explain why the aligned-column format made this awkward in the first place.

In [ ]:
# Write Challenge B here.

### Challenge C · Flatten and invert

Two small container puzzles:

1. Flatten `[[1, 2], [3, 4], [5]]` into `[1, 2, 3, 4, 5]` with one nested comprehension.
2. Invert `{"AE": "Emirates", "IN": "India"}` into `{"Emirates": "AE", "India": "IN"}` with
   one dict comprehension.

In [ ]:
# Write Challenge C here.

<details>
<summary><strong>Open solutions only after attempting the exercises</strong></summary>

## 11 · Solutions

### Exercises 1-3

```python
def rectangle_area(width, height):
    """Return the area of a rectangle."""
    return width * height


def apply_discount(price, percent=10):
    """Return price reduced by percent."""
    return price * (1 - percent / 100)


def min_max(values):
    """Return the smallest and largest value in a list."""
    smallest = values[0]
    largest = values[0]
    for value in values:
        if value < smallest:
            smallest = value
        if value > largest:
            largest = value
    return smallest, largest
```

### Exercises 4-6

```python
exercise_4 = "global"       # the local assignment shadowed it; the global never changed

exercise_5 = temps[::2]

exercise_6 = sorted(temps)[-1]    # sorted() returns a new list, so temps is untouched
```

### Exercises 7-9

```python
exercise_7 = {}
for letter in "mississippi":
    exercise_7[letter] = exercise_7.get(letter, 0) + 1

exercise_8 = sorted(set(raw_words))

exercise_9 = [1, 2, 3]      # y is an alias, so the append is visible through x
```

### Exercises 10-11

```python
exercise_10 = [n ** 2 for n in range(1, 11) if n % 2 == 0]

exercise_11a = {name: len(name) for name in people}
exercise_11b = sorted(people, key=len, reverse=True)
```

Note on 11b: `sorted` is **stable**, so `"turing"` and `"hopper"` (both length 6) keep their
original relative order.

### Project extensions

```python
def best_day(minutes):
    """Return the largest single day's minutes, or 0 for an empty list."""
    if not minutes:
        return 0
    return max(minutes)


def class_average(report):
    """Return the mean of every learner's average, rounded to one decimal."""
    averages = [summary["average"] for summary in report.values()]
    if not averages:
        return 0.0
    return round(sum(averages) / len(averages), 1)


def needs_encouragement(report):
    """Return the set of names whose habit is 'start small'."""
    return {name for name, summary in report.items() if summary["habit"] == "start small"}
```

Why does `class_average` expect **36.2** and not 36.3? The three averages are 38.75, 60.0,
and 10.0, whose mean is exactly 36.25. Python's `round()` breaks an exact half by rounding to
the **even** digit, so `round(36.25, 1)` is `36.2`. This is deliberate: always rounding halves
up would bias a long column of numbers upward. Try `round(0.5)`, `round(1.5)`, and
`round(2.5)` to see the rule: `0`, `2`, `2`.

F-string formatting follows the same rule, so `f"{36.25:.1f}"` is also `36.2` — reaching for
display formatting does not escape it. If a project genuinely requires "always round halves
up", that needs the `decimal` module:

```python
from decimal import Decimal, ROUND_HALF_UP
Decimal("36.25").quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)   # 36.3
```

You will not need `decimal` in this course; recognising that `round()` has a rule worth
checking is the transferable lesson.

```python
# Extension 5: filter first, inside summarize()
def summarize(minutes, goal=30):
    """Return one learner's figures as a dict, ignoring negative entries."""
    clean = [m for m in minutes if m >= 0]
    average = average_minutes(clean)
    return {
        "days": len(clean),
        "total": sum(clean),
        "average": average,
        "habit": classify(average),
        "at_goal": days_at_goal(clean, goal),
    }
```

### Challenge A

```python
def top_word(words):
    """Return the most frequent word in a list."""
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1

    best_word = words[0]
    for word in counts:
        if counts[word] > counts[best_word]:
            best_word = word
    return best_word
```

`max(counts, key=counts.get)` works because iterating a dict yields its keys, and `key=`
applies `counts.get` to each key to get the value `max` should compare. `counts.get` is
passed **without** parentheses: the function itself, not its result.

### Challenge B

```python
def day_label(count):
    """Return '1 day' or 'N days' with correct grammar."""
    if count == 1:
        return "1 day"
    return f"{count} days"
```

It was awkward because `{s['days']:>3} days` reserved a fixed three characters for the
number and a constant word after it. Correct grammar makes the text length vary, so the
column width must now be applied to the whole label: `f"{day_label(s['days']):<9}"`.

### Challenge C

```python
flat = [value for row in [[1, 2], [3, 4], [5]] for value in row]

codes = {"AE": "Emirates", "IN": "India"}
inverted = {name: code for code, name in codes.items()}
```

In the nested comprehension the two `for` clauses appear in the same order you would write
them as nested loops: outer first, inner second.

</details>

## 12 · Exit ticket: prove you are ready for Module 03

A performance check, not a recognition quiz. Complete all three parts without copying a
nearby example.

**Pass rule:** all three parts must be independently correct.

- **A** — your function must return `0.0` for `[]` and `20.0` for `[10, 20, 30]`.
- **B** — write your prediction in the comment *before* running the cell, and it must match.
- **C** — the repaired function must **return** its result and must not share state between
  calls.

If one part is not yet correct, review the linked lesson part, change your work, and try
that part again.

### A · Write

Define `average_or_zero(values)`: return the mean of a list of numbers, or `0.0` when the
list is empty. Use a docstring. Review: [Part 1](index.html#functions).

In [ ]:
def average_or_zero(values):
    """Return the mean of values, or 0.0 when values is empty."""
    ...        # replace this line

check("Exit A · empty", average_or_zero([]), 0.0)
check("Exit A · normal", average_or_zero([10, 20, 30]), 20.0)

### B · Predict

Write down what the cell prints **before** running it. Review:
[Part 6](index.html#mutability).

In [ ]:
# My prediction:
#   first  = ___
#   second = ___
#   third  = ___

def add_row(row, table=None):
    if table is None:
        table = []
    table.append(row)
    return table

first = add_row([1])
second = add_row([2])

shared = [[0]]
third = shared.copy()
third[0].append(9)

print("first :", first)
print("second:", second)
print("third :", third, " shared:", shared)

### C · Repair

This function has **three** defects from this module: a mutable default argument, a print
where a return belongs, and an in-place sort whose result is discarded. Fix all three so the
checks pass. Review: [Part 2](index.html#arguments) and [Part 4](index.html#lists).

In [ ]:
# Broken version -- rewrite it below the line.
def ranked_scores(new_score, scores=[]):
    scores.append(new_score)
    scores = scores.sort()
    print(scores)

# ---- your repaired version (keep the same name) ----


check("Exit C · first call", ranked_scores(90, [70, 85]), [70, 85, 90])
check("Exit C · no shared state", ranked_scores(50), [50])
check("Exit C · still no shared state", ranked_scores(60), [60])

## 13 · Summary and self-assessment

Check an item only when you can do it **without copying the lesson example**:

- [ ] I can write a function with a docstring, parameters, and a return value.
- [ ] I can explain why a cell containing only a `def` produces no output.
- [ ] I can state the difference between a parameter and an argument.
- [ ] I can explain what a function without `return` gives back, and why it breaks arithmetic.
- [ ] I can use positional arguments, keyword arguments, and default values correctly.
- [ ] I can explain why a list must never be a default value.
- [ ] I can return several values and unpack them.
- [ ] I can predict a shadowing example and rewrite a global-mutating function as a pure one.
- [ ] I can index and slice lists and strings, including negative indices and steps.
- [ ] I can explain why `scores = scores.sort()` is a bug.
- [ ] I can choose between list, tuple, dict, and set and justify the choice.
- [ ] I can build a counting dictionary with `.get()`.
- [ ] I can use set operations to compare two collections.
- [ ] I can predict aliasing outcomes and explain `is` versus `==`.
- [ ] I can explain when `copy.deepcopy()` is needed.
- [ ] I can convert a loop to a comprehension and back, and name a case where the loop wins.
- [ ] I can use `sorted(..., key=...)` and read a `lambda`.
- [ ] My project passes all seven test cases and I can run `study_stats.py`.
- [ ] I completed at least eight guided exercises before viewing the solutions.
- [ ] I passed all three exit-ticket parts.

Return to the [lesson self-check](index.html#self-check), then continue to
[Module 03 · Errors, Files, Classes & Modules](../03-python-oop-errors-files/index.html).